In [0]:
import polars as pl
from pathlib import Path

TRACE=Path("data/75k_unstable/trace-merged-dangerously.txt")
SEQDETECTIVE=Path("data/75k_unstable/seq-detective-judgement-summary-all.txt")
FILTERING=Path("data/75k_unstable/stats-merged.csv")
OUTDIR=Path("Figures")

In [5]:
trace_stats=pl.read_csv(TRACE, separator="\t", has_header=True, null_values="-")
sd_judgement=pl.read_csv(SEQDETECTIVE, separator="\t", has_header=False, new_columns=['id', 'file1', 'file2', 'judgement1', 'judgement2', 'reason'])
filter_stats=pl.read_csv(FILTERING, has_header=True)

In [6]:
# is this OK?  TT could be LQ PE *or* SC-like
judgement_mapping={
    "B": "SE",
    "T": "SE",
    "BB": "PE",
    "TT": "PE",
    "TB": "T-filt",
    "BT": "T-filt"    
}
judgement_map = sd_judgement.with_columns(
    id=pl.col("id").str.strip_suffix("_subsample"),
    tech_group=pl.concat_str(pl.col("judgement1"), pl.col("judgement2"),ignore_nulls=True).replace(judgement_mapping)
).select(pl.col("id"), pl.col("tech_group"))

judgement_map.group_by(pl.col("tech_group")).len()

tech_group,len
str,u32
"""PE""",36819
"""T-filt""",14825
"""SE""",25544


In [7]:
# filter the step trace
# For most steps, status MUST be one of COMPLETED or CACHED
OK_status = ["COMPLETED", "CACHED"]
# For Download, however, we allow one example of a failed run for initial counting
trace_good = trace_stats.filter(pl.col("status").is_in(OK_status) |
        (pl.col("process").eq("download"))
    ).unique(
        subset=["process", "tag"], keep="last", maintain_order=True
    ).join(judgement_map, left_on="tag", right_on="id", how="inner")

# Host filter: take just filtering steps
filter_steps = ["download", "fastp", "kb_negative", "hisat2", "star", "bowtie2", "dedup", "gsnap"]
host_filter = trace_good.filter(pl.col("process").is_in(filter_steps))

In [8]:
run_counts = host_filter.group_by("tech_group").agg(
        starting=pl.col("process").filter(pl.col("process").eq("download")).len(),
        seq_detective=pl.col("process").filter(pl.col("process").eq("download") & pl.col("status").is_in(OK_status)).len(),
        fastp=pl.col("process").filter(pl.col("process").eq("fastp")).len(),
        kb_negative=pl.col("process").filter(pl.col("process").eq("kb_negative")).len(),
        hisat2=pl.col("process").filter(pl.col("process").eq("hisat2")).len(),
        star=pl.col("process").filter(pl.col("process").eq("star")).len(),
        bowtie2=pl.col("process").filter(pl.col("process").eq("bowtie2")).len(),
        dedup=pl.col("process").filter(pl.col("process").eq("dedup")).len(),
        gsnap=pl.col("process").filter(pl.col("process").eq("gsnap")).len(),
        final=pl.col("process").filter(pl.col("process").eq("gsnap")).len(),
    ).transpose(
        include_header=True, header_name="step", column_names="tech_group"
    ).with_columns(
        Total=pl.sum_horizontal(pl.col("SE"), pl.col("PE"), pl.col("T-filt"))
    )

run_counts

step,PE,SE,T-filt,Total
str,u32,u32,u32,u32
"""starting""",36819,25544,14825,77188
"""seq_detective""",36819,25544,14824,77187
"""fastp""",34985,24548,14804,74337
"""kb_negative""",34979,24545,14798,74322
"""hisat2""",34979,24545,14794,74318
"""star""",34979,24543,14789,74311
"""bowtie2""",34979,24541,14786,74306
"""dedup""",34936,24334,14687,73957
"""gsnap""",34936,24334,14687,73957


In [9]:
with pl.Config() as cfg:
    cfg.set_tbl_formatting('ASCII_MARKDOWN')
    print(repr(run_counts))

shape: (10, 5)
| step          | PE    | SE    | T-filt | Total |
| ---           | ---   | ---   | ---    | ---   |
| str           | u32   | u32   | u32    | u32   |
|---------------|-------|-------|--------|-------|
| starting      | 36819 | 25544 | 14825  | 77188 |
| seq_detective | 36819 | 25544 | 14824  | 77187 |
| fastp         | 34985 | 24548 | 14804  | 74337 |
| kb_negative   | 34979 | 24545 | 14798  | 74322 |
| hisat2        | 34979 | 24545 | 14794  | 74318 |
| star          | 34979 | 24543 | 14789  | 74311 |
| bowtie2       | 34979 | 24541 | 14786  | 74306 |
| dedup         | 34936 | 24334 | 14687  | 73957 |
| gsnap         | 34936 | 24334 | 14687  | 73957 |
| final         | 34936 | 24334 | 14687  | 73957 |


In [10]:
filter_stats_cols=["id", "starting_reads", "fastp_reads_before", "kallisto_reads_before", "hisat2_reads_before", "star_reads_before", "bowtie2_reads_before", "dedup_reads_before", "gsnap_reads_before", "final_reads"]
host_steps = filter_stats.select(
        pl.col(filter_stats_cols)
    ).unique(
        subset=["id"], keep="last", maintain_order=True
    ).join(judgement_map, left_on="id", right_on="id", how="inner")

host_steps.group_by(pl.col("tech_group")).agg(count=pl.col("id").len(), start=pl.col("starting_reads").sum(), final=pl.col("final_reads").sum())

tech_group,count,start,final
str,u32,i64,i64
"""PE""",34931,617983579652,4726787174
"""SE""",24332,639806648664,3859713061
"""T-filt""",14683,385705062428,3026491567


In [11]:
read_counts = host_steps.group_by("tech_group").agg(
        starting=pl.col("starting_reads").sum(),
        seq_detective=pl.col("fastp_reads_before").sum(),
        fastp=pl.col("kallisto_reads_before").sum(),
        kb_negative=pl.col("hisat2_reads_before").sum(),
        hisat2=pl.col("star_reads_before").sum(),
        star=pl.col("bowtie2_reads_before").sum(),
        bowtie2=pl.col("dedup_reads_before").sum(),
        dedup=pl.col("gsnap_reads_before").sum(),
        gsnap=pl.col("final_reads").sum(),
        final=pl.col("final_reads").sum(),
    ).transpose(
        include_header=True, header_name="step", column_names="tech_group"
    ).with_columns(
        Total=pl.sum_horizontal(pl.col("SE"), pl.col("PE"), pl.col("T-filt"))
    )

In [12]:
trace_final_runs = frozenset(host_filter.filter(pl.col("process").eq("gsnap"))["tag"])
actual_outputs_P=! ls -1 data/75k_unstable/host_mapping/unmapped_reads/Paired
actual_outputs_S=! ls -1 data/75k_unstable/host_mapping/unmapped_reads/Single
actual_outputs=frozenset(actual_outputs_S+actual_outputs_P)

filtering_steps_set=frozenset(host_steps['id'])
print("trace run ids WITHOUT filtering steps")
print(trace_final_runs.difference(filtering_steps_set))
print("filtering steps WITHOUT trace run ids")
print(filtering_steps_set.difference(trace_final_runs))
print()
print("trace run ids WITHOUT outputs")
print(trace_final_runs.difference(actual_outputs))
print("outputs WITHOUT trace run ids")
print(actual_outputs.difference(trace_final_runs))
print()
print("filtering steps & ~outputs")
print(filtering_steps_set.difference(actual_outputs))
print("outputs & ~filtering steps")
print(actual_outputs.difference(filtering_steps_set))

trace run ids WITHOUT filtering steps
frozenset({'SRR24883272', 'SRR19977541', 'ERR1427405', 'SRR13565181', 'SRR32942196', 'SRR8587699', 'SRR31642040', 'SRR17282626', 'SRR23290627', 'SRR11293496', 'SRR32941746'})
filtering steps WITHOUT trace run ids
frozenset()

trace run ids WITHOUT outputs
frozenset({'SRR32942196', 'SRR32941746'})
outputs WITHOUT trace run ids
frozenset()

filtering steps & ~outputs
frozenset()
outputs & ~filtering steps
frozenset({'SRR24883272', 'SRR19977541', 'ERR1427405', 'SRR13565181', 'SRR8587699', 'SRR31642040', 'SRR17282626', 'SRR23290627', 'SRR11293496'})


In [13]:
with pl.Config() as cfg:
    cfg.set_tbl_formatting('ASCII_MARKDOWN')
    print(repr(read_counts))

shape: (10, 5)
| step          | PE           | SE           | T-filt       | Total         |
| ---           | ---          | ---          | ---          | ---           |
| str           | i64          | i64          | i64          | i64           |
|---------------|--------------|--------------|--------------|---------------|
| starting      | 617983579652 | 639806648664 | 385705062428 | 1643495290744 |
| seq_detective | 617523432664 | 639806648664 | 385639285330 | 1642969366658 |
| fastp         | 539137056918 | 537249723003 | 304580496317 | 1380967276238 |
| kb_negative   | 97458142341  | 149439619674 | 91235137668  | 338132899683  |
| hisat2        | 23963486644  | 48442807600  | 18165153826  | 90571448070   |
| star          | 10472674274  | 13127029858  | 6095244703   | 29694948835   |
| bowtie2       | 9223255385   | 11041856456  | 4794328319   | 25059440160   |
| dedup         | 4930679536   | 3911802496   | 3047860193   | 11890342225   |
| gsnap         | 4726787174   | 3859

In [14]:
RUN_NAMEMAP={"SE": "se_runs", "PE": "pe_runs", "T-filt": "tfilt_runs", "Total": "total_runs"}
READ_NAMEMAP={"SE": "se_reads", "PE": "pe_reads", "T-filt": "tfilt_reads", "Total": "total_reads"}
wide_df = run_counts.rename(
        RUN_NAMEMAP
    ).join(
        read_counts.rename(READ_NAMEMAP),
        left_on="step", right_on="step"
    )

In [15]:
from great_tables import GT, loc, style
stats_table = (
    GT(wide_df)
    .tab_header("Zebrafish RNAquarium Run & Read Filtering")
    #.tab_spanner(label="PE", columns=pl.selectors.starts_with("PE"))
    .fmt_integer(columns=pl.selectors.ends_with("runs"))
    .fmt_number(columns=pl.selectors.ends_with("reads"), compact=True, pattern="{x}", scale_by=1, n_sigfig=3)
    .data_color(columns=pl.selectors.ends_with("reads"))
)
stats_table

GT(_tbl_data=shape: (10, 9)
┌────────────┬─────────┬─────────┬────────────┬───┬────────────┬───────────┬───────────┬───────────┐
│ step       ┆ pe_runs ┆ se_runs ┆ tfilt_runs ┆ … ┆ pe_reads   ┆ se_reads  ┆ tfilt_rea ┆ total_rea │
│ ---        ┆ ---     ┆ ---     ┆ ---        ┆   ┆ ---        ┆ ---       ┆ ds        ┆ ds        │
│ str        ┆ u32     ┆ u32     ┆ u32        ┆   ┆ i64        ┆ i64       ┆ ---       ┆ ---       │
│            ┆         ┆         ┆            ┆   ┆            ┆           ┆ i64       ┆ i64       │
╞════════════╪═════════╪═════════╪════════════╪═══╪════════════╪═══════════╪═══════════╪═══════════╡
│ starting   ┆ 36819   ┆ 25544   ┆ 14825      ┆ … ┆ 6179835796 ┆ 639806648 ┆ 385705062 ┆ 164349529 │
│            ┆         ┆         ┆            ┆   ┆ 52         ┆ 664       ┆ 428       ┆ 0744      │
│ seq_detect ┆ 36819   ┆ 25544   ┆ 14824      ┆ … ┆ 6175234326 ┆ 639806648 ┆ 385639285 ┆ 164296936 │
│ ive        ┆         ┆         ┆            ┆   ┆ 64         ┆ 664       ┆ 330       ┆ 6658      │
│ fastp      ┆ 34985   ┆ 24548   ┆ 14804      ┆ … ┆ 5391370569 ┆ 537249723 ┆ 304580496 ┆ 138096727 │
│            ┆         ┆         ┆            ┆   ┆ 18         ┆ 003       ┆ 317       ┆ 6238      │
│ kb_negativ ┆ 34979   ┆ 24545   ┆ 14798      ┆ … ┆ 9745814234 ┆ 149439619 ┆ 912351376 ┆ 338132899 │
│ e          ┆         ┆         ┆            ┆   ┆ 1          ┆ 674       ┆ 68        ┆ 683       │
│ hisat2     ┆ 34979   ┆ 24545   ┆ 14794      ┆ … ┆ 2396348664 ┆ 484428076 ┆ 181651538 ┆ 905714480 │
│            ┆         ┆         ┆            ┆   ┆ 4          ┆ 00        ┆ 26        ┆ 70        │
│ star       ┆ 34979   ┆ 24543   ┆ 14789      ┆ … ┆ 1047267427 ┆ 131270298 ┆ 609524470 ┆ 296949488 │
│            ┆         ┆         ┆            ┆   ┆ 4          ┆ 58        ┆ 3         ┆ 35        │
│ bowtie2    ┆ 34979   ┆ 24541   ┆ 14786      ┆ … ┆ 9223255385 ┆ 110418564 ┆ 479432831 ┆ 250594401 │
│            ┆         ┆         ┆            ┆   ┆            ┆ 56        ┆ 9         ┆ 60        │
│ dedup      ┆ 34936   ┆ 24334   ┆ 14687      ┆ … ┆ 4930679536 ┆ 391180249 ┆ 304786019 ┆ 118903422 │
│            ┆         ┆         ┆            ┆   ┆            ┆ 6         ┆ 3         ┆ 25        │
│ gsnap      ┆ 34936   ┆ 24334   ┆ 14687      ┆ … ┆ 4726787174 ┆ 385971306 ┆ 302649156 ┆ 116129918 │
│            ┆         ┆         ┆            ┆   ┆            ┆ 1         ┆ 7         ┆ 02        │
│ final      ┆ 34936   ┆ 24334   ┆ 14687      ┆ … ┆ 4726787174 ┆ 385971306 ┆ 302649156 ┆ 116129918 │
│            ┆         ┆         ┆            ┆   ┆            ┆ 1         ┆ 7         ┆ 02        │
└────────────┴─────────┴─────────┴────────────┴───┴────────────┴───────────┴───────────┴───────────┘, _body=<great_tables._gt_data.Body object at 0x7f97dabd0f20>, _boxhead=Boxhead([ColInfo(var='step', type=<ColInfoTypeEnum.default: 1>, column_label='step', column_align='left', column_width=None), ColInfo(var='pe_runs', type=<ColInfoTypeEnum.default: 1>, column_label='pe_runs', column_align='right', column_width=None), ColInfo(var='se_runs', type=<ColInfoTypeEnum.default: 1>, column_label='se_runs', column_align='right', column_width=None), ColInfo(var='tfilt_runs', type=<ColInfoTypeEnum.default: 1>, column_label='tfilt_runs', column_align='right', column_width=None), ColInfo(var='total_runs', type=<ColInfoTypeEnum.default: 1>, column_label='total_runs', column_align='right', column_width=None), ColInfo(var='pe_reads', type=<ColInfoTypeEnum.default: 1>, column_label='pe_reads', column_align='right', column_width=None), ColInfo(var='se_reads', type=<ColInfoTypeEnum.default: 1>, column_label='se_reads', column_align='right', column_width=None), ColInfo(var='tfilt_reads', type=<ColInfoTypeEnum.default: 1>, column_label='tfilt_reads', column_align='right', column_width=None), ColInfo(var='total_reads', type=<ColInfoTypeEnum.default: 1>, column_label='total_reads', column_align='right', column_width=None)]), _stub=<great_tables._g

In [3]:
from great_tables import GT, loc, style
totals_table = (
    GT(wide_df.select(pl.col("step"), pl.col("total_runs"), pl.col("total_reads")))
    .tab_header("Zebrafish RNAquarium Run & Read Filtering")
    .cols_label(
        step="Step",
        total_runs="Total Runs",
        total_reads="Total Reads",
    )
    .fmt_integer(columns=pl.selectors.ends_with("runs"))
    #.data_color(columns=pl.selectors.ends_with("runs"), palette="Dark2", domain=[0,77188])
    .fmt_number(columns=pl.selectors.ends_with("reads"), compact=True, pattern="{x}", scale_by=1, n_sigfig=3)
    .data_color(columns=pl.selectors.ends_with("reads"), palette="Blues", domain=[1000000000,1800000000000])
)
GT.write_raw_html(totals_table, OUTDIR / "run_read_totals_table.html")
totals_table

NameError: name 'wide_df' is not defined

In [2]:
# contig breakdown
contigs_df = pl.DataFrame({
        "step": ["rnaSPAdes", "NT Danio + human filter", "NT or NR hits", "&nbsp;&nbsp;&nbsp;&nbsp;NT BLAST hits", "&nbsp;&nbsp;&nbsp;&nbsp;NR Diamond hits", "BBDuk flagged removed", "&nbsp;&nbsp;&nbsp;&nbsp;Dark Matter (not in NT or NR)"],
        "contigs": [
            89.8 * 10**6,
            48.0 * 10**6,
            37.9 * 10**6,
            30.8 * 10**6,
            28.7 * 10**6,            
            34.7 * 10**6,
            10.1 * 10**6
        ]
    })
contigs_table = (
    GT(contigs_df)
    .tab_header("Zebrafish RNAquarium Contig Breakdown")
    .cols_label(
        step="Part II Step",
        contigs="Total Contigs",
    )
    .tab_style(style=style.text(weight='normal'), locations=loc.body(columns=pl.col("step"),rows=[2, 3, 6]))
    .fmt_number(columns=pl.col("contigs"), compact=True, pattern="{x}", scale_by=1, n_sigfig=3)
    .data_color(columns=pl.col("contigs"), palette="Oranges", domain=[1000000, 100000000])
)
GT.write_raw_html(contigs_table, OUTDIR / "contigs_breakdown_table.html")
contigs_table

NameError: name 'pl' is not defined

NameError: name 'pl' is not defined